### Optimized ANN Model


In [1]:
import numpy as np
import pandas as pd
from scipy.sparse import load_npz
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam, RMSprop, SGD
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

I0000 00:00:1787562934.486284  810092 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1787562934.553966  810092 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1787562936.083639  810092 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
X_train_tfidf = load_npz("../models/X_train_tfidf.npz")
X_val_tfidf = load_npz("../models/X_val_tfidf.npz")
X_test_tfidf = load_npz("../models/X_test_tfidf.npz")

# Labels
y_train = np.load("../models/y_train.npy")
y_val = np.load("../models/y_val.npy")
y_test = np.load("../models/y_test.npy")

In [3]:
ann_results = []

def evaluate_ann(model, experiment_name):

    y_prob = model.predict(X_test_tfidf, verbose=0).ravel()

    y_pred = (y_prob >= 0.5).astype(int)

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)

    ann_results.append({
        "Experiment": experiment_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "ROC-AUC": roc_auc
    })

    print(f"\n{experiment_name}")
    print("-" * 40)
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print(f"ROC-AUC  : {roc_auc:.4f}")

In [4]:
def save_ann_results():

    results_df = pd.DataFrame(ann_results)

    results_df.to_csv(
        "../models/ann_optimization_results.csv",
        index=False
    )

    return results_df

In [5]:
from tensorflow.keras.models import load_model

ann_baseline = load_model("../models/ann_model.keras")

evaluate_ann(
    ann_baseline,
    "Baseline ANN"
)

save_ann_results()

E0000 00:00:1787563167.629182  810092 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1787563167.629507  812060 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1787563167.642719  810092 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...



Baseline ANN
----------------------------------------
Accuracy : 0.8720
Precision: 0.8544
Recall   : 0.8979
F1 Score : 0.8757
ROC-AUC  : 0.9385


,Experiment,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Baseline ANN,0.872009,0.854448,0.897937,0.875653,0.938522


### Hidden Units

In [6]:
input_dim = X_train_tfidf.shape[1]

ann_hidden = Sequential([
    Dense(256, activation="relu", input_shape=(input_dim,)),
    Dense(128, activation="relu"),
    Dense(1, activation="sigmoid")
])

ann_hidden.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history_hidden = ann_hidden.fit(
    X_train_tfidf,
    y_train,
    validation_data=(X_val_tfidf, y_val),
    epochs=10,
    batch_size=32,
    verbose=1
)


/home/aximsoft/snap/code/258/.local/share/virtualenvs/Sentiment_Analaysis-4UtJgnk7/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 55s 50ms/step - accuracy: 0.8705 - loss: 0.3032 - val_accuracy: 0.8945 - val_loss: 0.2534
Epoch 2/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 53s 49ms/step - accuracy: 0.9509 - loss: 0.1306 - val_accuracy: 0.8853 - val_loss: 0.3021
Epoch 3/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 54s 50ms/step - accuracy: 0.9821 - loss: 0.0446 - val_accuracy: 0.8868 - val_loss: 0.4489
Epoch 4/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 53s 49ms/step - accuracy: 0.9963 - loss: 0.0107 - val_accuracy: 0.8822 - val_loss: 0.7149
Epoch 5/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 55s 50ms/step - accuracy: 0.9994 - loss: 0.0024 - val_accuracy: 0.8860 - val_loss: 0.9000
Epoch 6/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 54s 50ms/step - accuracy: 0.9996 - loss: 0.0016 - val_accuracy: 0.8836 - val_loss: 1.0204
Epoch 7/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 54s 50ms/step - accuracy: 0.9995 - loss: 0.0015 - val_accuracy: 0.8743 - val_loss: 1.1408
Epoch 8/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 53s 48ms/step - accuracy: 0.9995 -

In [7]:
evaluate_ann(
    ann_hidden,
    "ANN - Hidden Units"
)



ANN - Hidden Units
----------------------------------------
Accuracy : 0.8770
Precision: 0.8613
Recall   : 0.8998
F1 Score : 0.8801
ROC-AUC  : 0.9444


In [8]:
ann_hidden.save("../models/ann_hidden_units.keras")
save_ann_results()

,Experiment,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Baseline ANN,0.872009,0.854448,0.897937,0.875653,0.938522
1,ANN - Hidden Units,0.876983,0.861282,0.899812,0.880126,0.944409


### Dropout

In [9]:
ann_dropout = Sequential([
    Dense(128, activation="relu", input_shape=(input_dim,)),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dropout(0.3),
    Dense(1, activation="sigmoid")
])

ann_dropout.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history_dropout = ann_dropout.fit(
    X_train_tfidf,
    y_train,
    validation_data=(X_val_tfidf, y_val),
    epochs=10,
    batch_size=32,
    verbose=1
)

/home/aximsoft/snap/code/258/.local/share/virtualenvs/Sentiment_Analaysis-4UtJgnk7/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 29s 26ms/step - accuracy: 0.8680 - loss: 0.3100 - val_accuracy: 0.8958 - val_loss: 0.2512
Epoch 2/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 28s 25ms/step - accuracy: 0.9472 - loss: 0.1429 - val_accuracy: 0.8935 - val_loss: 0.2928
Epoch 3/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 28s 26ms/step - accuracy: 0.9763 - loss: 0.0668 - val_accuracy: 0.8843 - val_loss: 0.3889
Epoch 4/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 28s 26ms/step - accuracy: 0.9926 - loss: 0.0232 - val_accuracy: 0.8829 - val_loss: 0.5447
Epoch 5/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 28s 26ms/step - accuracy: 0.9971 - loss: 0.0086 - val_accuracy: 0.8830 - val_loss: 0.6638
Epoch 6/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 27s 25ms/step - accuracy: 0.9987 - loss: 0.0044 - val_accuracy: 0.8837 - val_loss: 0.8072
Epoch 7/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 27s 25ms/step - accuracy: 0.9989 - loss: 0.0034 - val_accuracy: 0.8835 - val_loss: 0.8784
Epoch 8/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 27s 25ms/step - accuracy: 0.9987 -

In [10]:
evaluate_ann(
    ann_dropout,
    "ANN - Dropout"
)



ANN - Dropout
----------------------------------------
Accuracy : 0.8729
Precision: 0.8640
Recall   : 0.8864
F1 Score : 0.8750
ROC-AUC  : 0.9437


In [11]:
ann_dropout.save("../models/ann_dropout.keras")
save_ann_results()

,Experiment,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Baseline ANN,0.872009,0.854448,0.897937,0.875653,0.938522
1,ANN - Hidden Units,0.876983,0.861282,0.899812,0.880126,0.944409
2,ANN - Dropout,0.872950,0.863969,0.886418,0.875050,0.943730


### Batch Normalization

In [12]:
ann_batchnorm = Sequential([
    Dense(128, activation="relu", input_shape=(input_dim,)),
    BatchNormalization(),
    Dense(64, activation="relu"),
    BatchNormalization(),
    Dense(1, activation="sigmoid")
])

ann_batchnorm.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history_batchnorm = ann_batchnorm.fit(
    X_train_tfidf,
    y_train,
    validation_data=(X_val_tfidf, y_val),
    epochs=10,
    batch_size=32,
    verbose=1
)

/home/aximsoft/snap/code/258/.local/share/virtualenvs/Sentiment_Analaysis-4UtJgnk7/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 29s 25ms/step - accuracy: 0.8546 - loss: 0.3322 - val_accuracy: 0.8935 - val_loss: 0.2619
Epoch 2/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 27s 25ms/step - accuracy: 0.9633 - loss: 0.1022 - val_accuracy: 0.8820 - val_loss: 0.3407
Epoch 3/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 27s 25ms/step - accuracy: 0.9851 - loss: 0.0426 - val_accuracy: 0.8794 - val_loss: 0.4445
Epoch 4/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 27s 25ms/step - accuracy: 0.9860 - loss: 0.0390 - val_accuracy: 0.8794 - val_loss: 0.4709
Epoch 5/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 29s 27ms/step - accuracy: 0.9871 - loss: 0.0373 - val_accuracy: 0.8825 - val_loss: 0.4926
Epoch 6/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 27s 25ms/step - accuracy: 0.9900 - loss: 0.0288 - val_accuracy: 0.8837 - val_loss: 0.5474
Epoch 7/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 27s 25ms/step - accuracy: 0.9918 - loss: 0.0254 - val_accuracy: 0.8825 - val_loss: 0.5427
Epoch 8/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 27s 25ms/step - accuracy: 0.9941 -

In [13]:
evaluate_ann(
    ann_batchnorm,
    "ANN - Batch Normalization"
)



ANN - Batch Normalization
----------------------------------------
Accuracy : 0.8738
Precision: 0.8776
Recall   : 0.8698
F1 Score : 0.8737
ROC-AUC  : 0.9482


In [14]:
ann_batchnorm.save("../models/ann_batchnorm.keras")
save_ann_results()

,Experiment,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Baseline ANN,0.872009,0.854448,0.897937,0.875653,0.938522
1,ANN - Hidden Units,0.876983,0.861282,0.899812,0.880126,0.944409
2,ANN - Dropout,0.872950,0.863969,0.886418,0.875050,0.943730
3,ANN - Batch Normalization,0.873756,0.877568,0.869810,0.873671,0.948184


### Different Optimizer

In [15]:
ann_optimizer = Sequential([
    Dense(128, activation="relu", input_shape=(input_dim,)),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

ann_optimizer.compile(
    optimizer=RMSprop(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history_optimizer = ann_optimizer.fit(
    X_train_tfidf,
    y_train,
    validation_data=(X_val_tfidf, y_val),
    epochs=10,
    batch_size=32,
    verbose=1
)

/home/aximsoft/snap/code/258/.local/share/virtualenvs/Sentiment_Analaysis-4UtJgnk7/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 24s 21ms/step - accuracy: 0.8752 - loss: 0.3050 - val_accuracy: 0.9004 - val_loss: 0.2489
Epoch 2/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 22s 20ms/step - accuracy: 0.9291 - loss: 0.1858 - val_accuracy: 0.8966 - val_loss: 0.2546
Epoch 3/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 21s 20ms/step - accuracy: 0.9512 - loss: 0.1420 - val_accuracy: 0.8929 - val_loss: 0.2846
Epoch 4/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 23s 21ms/step - accuracy: 0.9648 - loss: 0.1071 - val_accuracy: 0.8903 - val_loss: 0.3100
Epoch 5/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 24s 22ms/step - accuracy: 0.9768 - loss: 0.0767 - val_accuracy: 0.8864 - val_loss: 0.3808
Epoch 6/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 23s 21ms/step - accuracy: 0.9850 - loss: 0.0503 - val_accuracy: 0.8805 - val_loss: 0.4383
Epoch 7/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 22s 20ms/step - accuracy: 0.9919 - loss: 0.0290 - val_accuracy: 0.8794 - val_loss: 0.5280
Epoch 8/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 22s 20ms/step - accuracy: 0.9956 -

In [16]:
evaluate_ann(
    ann_optimizer,
    "ANN - RMSprop Optimizer"
)



ANN - RMSprop Optimizer
----------------------------------------
Accuracy : 0.8731
Precision: 0.8562
Recall   : 0.8979
F1 Score : 0.8766
ROC-AUC  : 0.9450


In [17]:
ann_optimizer.save("../models/ann_rmsprop.keras")
save_ann_results()

,Experiment,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Baseline ANN,0.872009,0.854448,0.897937,0.875653,0.938522
1,ANN - Hidden Units,0.876983,0.861282,0.899812,0.880126,0.944409
2,ANN - Dropout,0.872950,0.863969,0.886418,0.875050,0.943730
3,ANN - Batch Normalization,0.873756,0.877568,0.869810,0.873671,0.948184
4,ANN - RMSprop Optimizer,0.873084,0.856194,0.897937,0.876569,0.945031


### Learning Rate

In [19]:
ann_lr = Sequential([
    Dense(128, activation="relu", input_shape=(input_dim,)),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

ann_lr.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history_lr = ann_lr.fit(
    X_train_tfidf,
    y_train,
    validation_data=(X_val_tfidf, y_val),
    epochs=10,
    batch_size=32,
    verbose=1
)


/home/aximsoft/snap/code/258/.local/share/virtualenvs/Sentiment_Analaysis-4UtJgnk7/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 31s 27ms/step - accuracy: 0.8477 - loss: 0.4264 - val_accuracy: 0.9035 - val_loss: 0.2516
Epoch 2/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 29s 27ms/step - accuracy: 0.9307 - loss: 0.1887 - val_accuracy: 0.9071 - val_loss: 0.2341
Epoch 3/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 29s 26ms/step - accuracy: 0.9563 - loss: 0.1312 - val_accuracy: 0.9021 - val_loss: 0.2505
Epoch 4/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 29s 27ms/step - accuracy: 0.9725 - loss: 0.0931 - val_accuracy: 0.8945 - val_loss: 0.2842
Epoch 5/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 29s 27ms/step - accuracy: 0.9837 - loss: 0.0642 - val_accuracy: 0.8894 - val_loss: 0.3245
Epoch 6/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 29s 27ms/step - accuracy: 0.9912 - loss: 0.0417 - val_accuracy: 0.8857 - val_loss: 0.3739
Epoch 7/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 29s 27ms/step - accuracy: 0.9958 - loss: 0.0257 - val_accuracy: 0.8820 - val_loss: 0.4269
Epoch 8/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 29s 27ms/step - accuracy: 0.9982 -

In [20]:
evaluate_ann(
    ann_lr,
    "ANN - Learning Rate 0.0001"
)



ANN - Learning Rate 0.0001
----------------------------------------
Accuracy : 0.8649
Precision: 0.8641
Recall   : 0.8671
F1 Score : 0.8656
ROC-AUC  : 0.9417


In [21]:
ann_lr.save("../models/ann_learning_rate.keras")
save_ann_results()

,Experiment,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Baseline ANN,0.872009,0.854448,0.897937,0.875653,0.938522
1,ANN - Hidden Units,0.876983,0.861282,0.899812,0.880126,0.944409
2,ANN - Dropout,0.872950,0.863969,0.886418,0.875050,0.943730
3,ANN - Batch Normalization,0.873756,0.877568,0.869810,0.873671,0.948184
4,ANN - RMSprop Optimizer,0.873084,0.856194,0.897937,0.876569,0.945031
5,ANN - Learning Rate 0.0001,0.864883,0.864122,0.867131,0.865624,0.941746


### Batch Size

In [22]:
ann_batchsize = Sequential([
    Dense(128, activation="relu", input_shape=(input_dim,)),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

ann_batchsize.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history_batchsize = ann_batchsize.fit(
    X_train_tfidf,
    y_train,
    validation_data=(X_val_tfidf, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)


/home/aximsoft/snap/code/258/.local/share/virtualenvs/Sentiment_Analaysis-4UtJgnk7/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 17s 29ms/step - accuracy: 0.8676 - loss: 0.3103 - val_accuracy: 0.9007 - val_loss: 0.2469
Epoch 2/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 16s 29ms/step - accuracy: 0.9495 - loss: 0.1400 - val_accuracy: 0.8852 - val_loss: 0.3034
Epoch 3/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 16s 29ms/step - accuracy: 0.9742 - loss: 0.0742 - val_accuracy: 0.8817 - val_loss: 0.3642
Epoch 4/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 16s 29ms/step - accuracy: 0.9895 - loss: 0.0292 - val_accuracy: 0.8785 - val_loss: 0.5313
Epoch 5/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 16s 29ms/step - accuracy: 0.9966 - loss: 0.0097 - val_accuracy: 0.8787 - val_loss: 0.6850
Epoch 6/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 16s 29ms/step - accuracy: 0.9992 - loss: 0.0025 - val_accuracy: 0.8761 - val_loss: 0.8314
Epoch 7/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 16s 29ms/step - accuracy: 0.9996 - loss: 8.9412e-04 - val_accuracy: 0.8765 - val_loss: 0.9225
Epoch 8/10
543/543 ━━━━━━━━━━━━━━━━━━━━ 16s 29ms/step - accuracy: 0.9999 - loss: 5.111

In [23]:
evaluate_ann(
    ann_batchsize,
    "ANN - Batch Size 64"
)



ANN - Batch Size 64
----------------------------------------
Accuracy : 0.8703
Precision: 0.8681
Recall   : 0.8744
F1 Score : 0.8712
ROC-AUC  : 0.9393


In [24]:
ann_batchsize.save("../models/ann_batchsize.keras")
save_ann_results()

,Experiment,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Baseline ANN,0.872009,0.854448,0.897937,0.875653,0.938522
1,ANN - Hidden Units,0.876983,0.861282,0.899812,0.880126,0.944409
2,ANN - Dropout,0.872950,0.863969,0.886418,0.875050,0.943730
3,ANN - Batch Normalization,0.873756,0.877568,0.869810,0.873671,0.948184
4,ANN - RMSprop Optimizer,0.873084,0.856194,0.897937,0.876569,0.945031
5,ANN - Learning Rate 0.0001,0.864883,0.864122,0.867131,0.865624,0.941746
6,ANN - Batch Size 64,0.870261,0.868085,0.874364,0.871213,0.939287


### Early Stopping

In [25]:
ann_earlystop = Sequential([
    Dense(128, activation="relu", input_shape=(input_dim,)),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

ann_earlystop.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

history_earlystop = ann_earlystop.fit(
    X_train_tfidf,
    y_train,
    validation_data=(X_val_tfidf, y_val),
    epochs=10,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)


/home/aximsoft/snap/code/258/.local/share/virtualenvs/Sentiment_Analaysis-4UtJgnk7/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 30s 27ms/step - accuracy: 0.8717 - loss: 0.3037 - val_accuracy: 0.8994 - val_loss: 0.2519
Epoch 2/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 29s 27ms/step - accuracy: 0.9485 - loss: 0.1400 - val_accuracy: 0.8871 - val_loss: 0.2957
Epoch 3/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 30s 27ms/step - accuracy: 0.9766 - loss: 0.0628 - val_accuracy: 0.8818 - val_loss: 0.4141


In [26]:
evaluate_ann(
    ann_earlystop,
    "ANN - Early Stopping"
)



ANN - Early Stopping
----------------------------------------
Accuracy : 0.8943
Precision: 0.8926
Recall   : 0.8974
F1 Score : 0.8950
ROC-AUC  : 0.9605


In [27]:
ann_earlystop.save("../models/ann_early_stopping.keras")
save_ann_results()

,Experiment,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Baseline ANN,0.872009,0.854448,0.897937,0.875653,0.938522
1,ANN - Hidden Units,0.876983,0.861282,0.899812,0.880126,0.944409
2,ANN - Dropout,0.872950,0.863969,0.886418,0.875050,0.943730
3,ANN - Batch Normalization,0.873756,0.877568,0.869810,0.873671,0.948184
4,ANN - RMSprop Optimizer,0.873084,0.856194,0.897937,0.876569,0.945031
5,ANN - Learning Rate 0.0001,0.864883,0.864122,0.867131,0.865624,0.941746
6,ANN - Batch Size 64,0.870261,0.868085,0.874364,0.871213,0.939287
7,ANN - Early Stopping,0.894326,0.892619,0.897402,0.895004,0.960532


### Learning Rate Scheduling

In [28]:
ann_scheduler = Sequential([
    Dense(128, activation="relu", input_shape=(input_dim,)),
    Dense(64, activation="relu"),
    Dense(1, activation="sigmoid")
])

ann_scheduler.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=1,
    min_lr=1e-6
)

history_scheduler = ann_scheduler.fit(
    X_train_tfidf,
    y_train,
    validation_data=(X_val_tfidf, y_val),
    epochs=10,
    batch_size=32,
    callbacks=[reduce_lr],
    verbose=1
)



/home/aximsoft/snap/code/258/.local/share/virtualenvs/Sentiment_Analaysis-4UtJgnk7/lib/python3.12/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 31s 28ms/step - accuracy: 0.8728 - loss: 0.3035 - val_accuracy: 0.8984 - val_loss: 0.2516 - learning_rate: 0.0010
Epoch 2/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 29s 27ms/step - accuracy: 0.9486 - loss: 0.1376 - val_accuracy: 0.8844 - val_loss: 0.2955 - learning_rate: 0.0010
Epoch 3/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 28s 26ms/step - accuracy: 0.9834 - loss: 0.0456 - val_accuracy: 0.8835 - val_loss: 0.4231 - learning_rate: 5.0000e-04
Epoch 4/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 29s 26ms/step - accuracy: 0.9952 - loss: 0.0132 - val_accuracy: 0.8837 - val_loss: 0.5283 - learning_rate: 2.5000e-04
Epoch 5/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 29s 27ms/step - accuracy: 0.9972 - loss: 0.0069 - val_accuracy: 0.8832 - val_loss: 0.5848 - learning_rate: 1.2500e-04
Epoch 6/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 28s 26ms/step - accuracy: 0.9982 - loss: 0.0050 - val_accuracy: 0.8840 - val_loss: 0.6339 - learning_rate: 6.2500e-05
Epoch 7/10
1085/1085 ━━━━━━━━━━━━━━━━━━━━ 29s 26

In [29]:
evaluate_ann(
    ann_scheduler,
    "ANN - Learning Rate Scheduling"
)



ANN - Learning Rate Scheduling
----------------------------------------
Accuracy : 0.8775
Precision: 0.8757
Recall   : 0.8811
F1 Score : 0.8784
ROC-AUC  : 0.9489


In [30]:
ann_scheduler.save("../models/ann_lr_scheduler.keras")
save_ann_results()

,Experiment,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,Baseline ANN,0.872009,0.854448,0.897937,0.875653,0.938522
1,ANN - Hidden Units,0.876983,0.861282,0.899812,0.880126,0.944409
2,ANN - Dropout,0.872950,0.863969,0.886418,0.875050,0.943730
3,ANN - Batch Normalization,0.873756,0.877568,0.869810,0.873671,0.948184
4,ANN - RMSprop Optimizer,0.873084,0.856194,0.897937,0.876569,0.945031
5,ANN - Learning Rate 0.0001,0.864883,0.864122,0.867131,0.865624,0.941746
6,ANN - Batch Size 64,0.870261,0.868085,0.874364,0.871213,0.939287
7,ANN - Early Stopping,0.894326,0.892619,0.897402,0.895004,0.960532
8,ANN - Learning Rate Scheduling,0.877521,0.875666,0.881061,0.878355,0.948879


In [32]:
ann_results_df.to_csv(
    "../models/ann_optimization_results.csv",
    index=False
)